# Adaptive SERSEM category weighting — frozen StarCoder2

Our supervised extension of this repository's SERSEM reconstruction; no author-equivalence claim. Baseline/neighbourhood code and saved results stay unchanged. Both modes first validate the existing complete 1,600 shadow / 200 fitting / 200 calibration / 1,000 held-out manifests. `MODE="smoke"` then selects reproducible, balanced language/membership subsets of 80/20/20/30 IDs using the baseline seed. `MODE="full"` uses every original ID. No split generation is available. Smoke metrics are preliminary and resolution-limited. All target inference runs manually here on Kaggle GPU; only the tiny adaptive optimizer runs on CPU. No LUMIA or StarCoder2 training.

Stages: load saved manifests → extract/restore compact token features → verify static scores → CPU fitting → validation/freeze → calibration-only threshold → exact held-out comparison. Held-out labels never enter optimization, early stopping or threshold selection. Static verification checks saved grey-box scores, not adaptive quality.


In [ ]:
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
UPLOAD_SOURCE = INPUT_ROOT / "EDIT-adaptive-source-dataset"
UPLOAD_ZIP = UPLOAD_SOURCE / "adaptive_kaggle_source.zip"
PROJECT_ROOT = WORK_ROOT / "adaptive-sersem-source-v2"
BASELINE_RUN = ""  # auto-discover one run in /kaggle/input; set explicitly if ambiguous
PATH_OVERRIDES = dict(shadow="", validation="", calibration="", heldout="", scores="", config="")
# Optional per-file overrides accept arbitrary mounted paths; no filenames are guessed.
MODE = "smoke"  # "smoke" or "full"
SMOKE_SIZES = dict(shadow=80, validation=20, calibration=20, heldout=30)
RUN_ID = f"adaptive-sersem-{MODE}-v1"  # distinct smoke/full outputs and checkpoints
OUT = WORK_ROOT / "adaptive_outputs" / RUN_ID
RESUME_FROM = ""  # trusted saved adaptive OUT; preserve the same config/source/manifests
RUN_EXTRACTION = True  # False permits cache-only CPU work; missing cached IDs then fail
FEATURE_CACHE_PATH = ""  # optional attached token_features.sqlite from a prior compatible run
FEATURE_CACHE_READ_ONLY = True  # used only when FEATURE_CACHE_PATH is supplied
FEATURE_REUSE_FROM = ""  # optional smoke token_features.sqlite to import into this run
# Full after smoke: keep FEATURE_CACHE_PATH="", use FEATURE_REUSE_FROM for partial cache reuse.
# Never point RESUME_FROM at a smoke output when starting a full run.
MODEL_CACHE_DIR = ""  # empty inherits baseline model cache; edit if moved
MODEL_LOCAL_FILES_ONLY = True
ALLOW_MODEL_DOWNLOAD = False  # optional Kaggle-only setup; attach existing target cache by default
DATASET_CACHE_DIR = WORK_ROOT / "hf_cache/datasets"  # set mounted HF Arrow cache if relocated
SOURCE_ACCESS = "cached"  # cached | streaming; streaming explicitly uses network
TRAINING_OPTIONS = dict(max_epochs=500, learning_rate=.03, patience=40, min_delta=1e-6,
                        regularization=.01, calibration_l2=1e-4, initial_scale=4.,
                        initial_noise=.001, gradient_clip=10.)  # seed inherited from baseline
SESSION_BUDGET_MINUTES = 480
DISK_RESERVE_GIB = 3.
WORK_ROOT.mkdir(parents=True, exist_ok=True)


## 1. Verify and load the source package
Upload the rebuilt `adaptive_kaggle_source.zip` as a Kaggle Dataset, or attach its extracted files. Set `UPLOAD_SOURCE`. This verifies allowlisted code before imports; an existing mismatched source directory is rejected. The notebook does not change baseline outputs.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
SOURCE_MANIFEST_SHA256 = 'abc5f0fb886491f3ad72488c733446f63a90c1a8f5f0bcc15e5c7656da23d838'
if not (PROJECT_ROOT / "adaptive_token_attack/source_manifest.json").is_file():
    if PROJECT_ROOT.exists():
        raise FileExistsError("Incomplete PROJECT_ROOT; select a fresh directory")
    if (UPLOAD_SOURCE / "adaptive_token_attack/source_manifest.json").is_file():
        source_manifest_path = UPLOAD_SOURCE / "adaptive_token_attack/source_manifest.json"
        if hashlib.sha256(source_manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
            raise ValueError("Outdated adaptive source package")
        source_manifest = json.loads(source_manifest_path.read_text())
        PROJECT_ROOT.mkdir(parents=True)
        for name in list(source_manifest) + ["adaptive_token_attack/source_manifest.json"]:
            destination = (PROJECT_ROOT / name).resolve()
            if not destination.is_relative_to(PROJECT_ROOT.resolve()):
                raise ValueError("Unsafe source path")
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(UPLOAD_SOURCE / name, destination)
    else:
        with zipfile.ZipFile(UPLOAD_ZIP) as archive:
            for entry in archive.infolist():
                if not (PROJECT_ROOT / entry.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe archive path")
            archive.extractall(PROJECT_ROOT)
manifest_path = PROJECT_ROOT / "adaptive_token_attack/source_manifest.json"
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
    raise ValueError("Source manifest mismatch; use a fresh PROJECT_ROOT and rebuilt package")
source_manifest = json.loads(manifest_path.read_text())
for name, expected in source_manifest.items():
    if hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest() != expected:
        raise ValueError("Source mismatch: " + name)
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "SERSEM-Poisoned-Chalice-Competition-2026"
sys.path.insert(0, str(UPSTREAM / "code"))


In [ ]:
# Install the existing pinned baseline dependencies; preserve Kaggle's numerical stack.
from shared.kaggle_setup import install_dependencies, validate_environment
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
packages = ["libenchant-2-2", "hunspell-en-us"]
status = subprocess.run(["dpkg-query", "-W", "-f=${Status}\n", *packages], capture_output=True, text=True)
if status.returncode != 0 or status.stdout.splitlines().count("install ok installed") != len(packages):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *packages], check=True)
install_dependencies(PROJECT_ROOT / "shared/requirements-kaggle.txt", WORK_ROOT)
environment = validate_environment(PROJECT_ROOT / "shared/requirements-kaggle.txt")


## 2. Load exact saved partitions and validate provenance
Both modes require complete original manifests of 1,600/200/200/1,000 rows before any subset selection. Smoke then enforces `SMOKE_SIZES`; full preserves all rows and their original order. IDs **and content hashes** must be disjoint; shadow's original LUMIA internal role labels are accepted as one development pool. Missing files are listed together. Multiple discovered run directories require explicit `BASELINE_RUN`; no automatic resampling or held-out training is possible.


In [ ]:
import copy
import numpy as np
import pandas as pd
from adaptive_token_attack.data import resolve_paths, load_run, frame_fingerprint, select_partitions
from adaptive_token_attack.features import FeatureStore, extraction_identity, extract_manifest, FrozenExtractor, load_frozen_target
from adaptive_token_attack.model import make_pool, fit, freeze, validation_loss
from adaptive_token_attack.evaluation import verify_static, calibrate, evaluate
from shared.resource_data import SourceStore
from shared.resource_pipeline import Monitor, restore_run
from shared.sersem_baseline import write_json
paths = resolve_paths(BASELINE_RUN, PATH_OVERRIDES, INPUT_ROOT)
original_partitions, saved_scores, base_cfg = load_run(paths)  # validates ALL original IDs and scores
partitions, execution = select_partitions(original_partitions, MODE, SMOKE_SIZES, seed=base_cfg["seed"])
if base_cfg["versions"] != environment["versions"]:
    raise ValueError("Restore the saved baseline numerical/API environment before extraction/comparison")
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("RUN_ID must be a directory name")
if OUT.resolve().is_relative_to(paths["config"].parent.resolve()) or any(p.resolve().is_relative_to(OUT.resolve()) for p in paths.values()):
    raise ValueError("Adaptive outputs must be separate from baseline input files")
OUT.mkdir(parents=True, exist_ok=True)
if RESUME_FROM:
    resume_config = json.loads((Path(RESUME_FROM) / "config.json").read_text())
    if resume_config.get("mode") != MODE:
        raise ValueError("RESUME_FROM must use the same mode; use FEATURE_REUSE_FROM to import only smoke features into full mode")
    restore_run(RESUME_FROM, OUT)
training_cfg = dict(TRAINING_OPTIONS, seed=base_cfg["seed"])
feature_identity = extraction_identity(base_cfg, PROJECT_ROOT)
experiment = dict(schema="adaptive-experiment-v2", mode=MODE, execution=execution, source_manifest_sha256=SOURCE_MANIFEST_SHA256,
                  baseline_config_sha256=hashlib.sha256(paths["config"].read_bytes()).hexdigest(),
                  saved_scores_sha256=hashlib.sha256(paths["scores"].read_bytes()).hexdigest(),
                  manifests={role:frame_fingerprint(frame) for role,frame in partitions.items()},
                  training=training_cfg, extraction=feature_identity)
if (OUT / "config.json").exists() and json.loads((OUT / "config.json").read_text()) != experiment:
    raise ValueError("Run identity changed; use a new RUN_ID")
write_json(OUT / "config.json", experiment)
write_json(OUT / "selection.json", execution)
write_json(OUT / "environment.json", environment)
write_json(OUT / "input_paths.json", {k:str(v) for k,v in paths.items()})
for role, frame in partitions.items():
    frame.to_parquet(OUT / (role + "_manifest.parquet"), index=False)
print("Mode:", MODE, "selected:", {role:len(frame) for role,frame in partitions.items()})
print("Complete originals:", {role:len(frame) for role,frame in original_partitions.items()})


## 3. Extract or restore token features (Kaggle GPU only)
Smoke defaults require up to **150** frozen target forwards; full requires up to **3,000**, one per uncached selected sample. Originals are validated but unselected rows are never sent to extraction. No probes, hidden states or vocabulary logits are cached. Each sample commits atomically. A complete cache skips model loading and source iteration. To reuse an attached cache in a new hyperparameter run, set `FEATURE_CACHE_PATH` to its `token_features.sqlite` (read-only by default) and choose a new `RUN_ID`. An incomplete read-only cache fails before target loading. To reuse the partial smoke cache for a full run, use a new full run ID, leave `RESUME_FROM` and `FEATURE_CACHE_PATH` empty, and set `FEATURE_REUSE_FROM` to the saved smoke `token_features.sqlite`. Compatible per-token rows are imported into the full run’s writable cache and only missing rows are inferred. No smoke weights, optimizer state or thresholds are imported. Incomplete extraction blocks later stages; failures are logged and raised. Existing scalar SERSEM checkpoints do not contain sufficient token features.

Attach the original HF dataset and target caches. No dataset reconstruction/preparation runs here. Set `SOURCE_ACCESS="streaming"` explicitly for remote source iteration. `ALLOW_MODEL_DOWNLOAD=True` permits downloading the pinned target on Kaggle if needed. Model/cache paths are operational; revision, dtype, context, numerical versions and source hashes are fixed.


In [ ]:
store = FeatureStore(FEATURE_CACHE_PATH or OUT / "token_features.sqlite", feature_identity,
                     read_only=bool(FEATURE_CACHE_PATH) and FEATURE_CACHE_READ_ONLY)
source_cfg = copy.deepcopy(base_cfg)
source_cfg["resources"].update(data_access=SOURCE_ACCESS, text_cache_gib=.1,
    session_budget_minutes=SESSION_BUDGET_MINUTES, disk_reserve_gib=DISK_RESERVE_GIB)
monitor = Monitor(source_cfg, OUT)
source_store = SourceStore(source_cfg, OUT, monitor)
all_samples = pd.concat(list(partitions.values()), ignore_index=True)  # selected rows only
if FEATURE_REUSE_FROM:
    reused = store.reuse_from(FEATURE_REUSE_FROM, all_samples)
    write_json(OUT / "imported_features.json", reused)
    print(reused)
# This factory is called only if a sample is absent from the feature cache.
def extractor_factory():
    import torch
    from shared.cache_setup import ensure_model_snapshot
    if not torch.cuda.is_available():
        raise RuntimeError("Missing token features require Kaggle GPU; cached-weight training remains CPU-compatible")
    monitor.guard()
    cache_dir = MODEL_CACHE_DIR or base_cfg["resources"]["model_cache_dir"]
    ensure_model_snapshot(base_cfg["model_id"], base_cfg["model_revision"], cache_dir,
                          ALLOW_MODEL_DOWNLOAD, DISK_RESERVE_GIB, WORK_ROOT, OUT / "model_cache_setup.json")
    model, tokenizer = load_frozen_target(base_cfg, cache_dir, MODEL_LOCAL_FILES_ONLY)
    return FrozenExtractor(model, tokenizer, base_cfg)
if RUN_EXTRACTION:
    summary = extract_manifest(all_samples, store, source_store, extractor_factory, monitor)
    write_json(OUT / "feature_extraction.json", dict(summary, mode=MODE))
    print(summary)
else:
    store.digest(all_samples)  # fail with a missing sample ID; never fall back to held-out data
# No model reference survives extract_manifest. Free its inactive CUDA allocations.
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 4. Reconstruct and verify original static scores
CPU reconstruction must agree with extraction-time GPU scores within 2e-6. Both must also agree with the **actual saved grey-box scores for the selected held-out IDs** (30 in default smoke, 1,000 in full): absolute tolerance 2e-6 for float32 or 2e-5 for float16/bfloat16, relative tolerance zero. Mismatches write diagnostics and raise; tolerances never auto-relax. Diagnose checkpoint, source text, dtype, tokenizer, backend, dependencies and weighting before proceeding. This is a fidelity gate, not an adaptive hyperparameter search.


In [ ]:
pools = {role:make_pool(frame, role, store) for role,frame in partitions.items()}
static_gate = verify_static(pools, saved_scores, store, base_cfg["dtype"], OUT, execution=execution)
print({k:v for k,v in static_gate.items() if k != "records"})


## 5. Fit category weights on CPU
Full-batch NumPy Adam learns eight category logits, one positive calibration scale and one bias. Stable `logaddexp` BCE is equivalent to BCEWithLogitsLoss. The objective adds KL(static prior || learned weights) and small calibration L2 penalties. Initialization is near the static prior. Validation BCE uses only the selected fitting pool (20 in default smoke; 200 in full) for early stopping. Weight fitting uses only the selected development pool (80 in smoke; 1,600 in full). The calibration/test labels are not passed to fitting.

Every epoch saves Adam moments, parameters, best parameters, patience state and history. Rerun this cell to resume exactly; changed scientific settings require a new run. No repeated target inference occurs.


In [ ]:
model_weights, complete = fit(pools["shadow"], pools["validation"], training_cfg, OUT, static_gate)
if not complete:
    raise RuntimeError("Training incomplete; resume before freezing")
print("Best validation BCE:", model_weights["best_validation_bce"])
print("Learned weights:", model_weights["weights"])


## 6. Validate and freeze the best checkpoint
Only the fitting pool is used. Freeze before inspecting adaptive held-out performance. Do not revise hyperparameters based on held-out results.


In [ ]:
frozen = freeze(model_weights, pools["validation"], OUT)
print("Frozen validation BCE:", frozen["frozen_validation_bce"])
print("Calibration scale (positive):", frozen["calibration_scale"])


## 7. Select the threshold on the separate calibration pool
Select a global raw-score threshold at empirical FPR <=1%. No calibration samples are used to optimize the weights or calibration scale/bias. Ties are handled conservatively; no finite eligible ROC threshold yields a threshold above the maximum score.


In [ ]:
threshold = calibrate(frozen, pools["calibration"], OUT)
print(json.dumps(threshold, indent=2))


## 8. Evaluate the selected frozen held-out IDs
Recompute ROC-AUC, empirical TPR at FPR <=1%, and per-language ROC reports for adaptive, static grey-box, saved LUMIA and saved baseline fusion on the same exact identities. Report the adaptive calibrated-threshold TPR/FPR separately. No fusion with the adaptive or neighbourhood attack is performed. Full coverage of the selected cohort is required; missing features/scores raise instead of silently filtering. Smoke evaluates only its selected 30 IDs, with all performance reports explicitly labeled preliminary and resolution-limited. Full evaluates the exact original 1,000 IDs.


In [ ]:
report = evaluate(frozen, threshold, pools["heldout"], saved_scores, OUT)
print(json.dumps(report, indent=2))
store.close()


## Save outputs and resume
Persist the **entire OUT directory** with Kaggle Save Version **with outputs**, including any SQLite journal, before ending a session. Next session attach saved adaptive outputs, the original saved SERSEM run and separate source/target caches; set the same `RUN_ID` and `RESUME_FROM` to the saved adaptive directory. Only operational locations/budgets may change on resume; mode, subset sizes and seed are bound into configuration/fitted-artifact identities. Smoke and full require separate run directories. Rerunning the extraction stage uses cached samples; fitting resumes exact Adam/early-stop state. Baseline files and its saved results are never modified.

Artifacts: `config.json`, `selection.json` (mode, seed, original and selected fingerprints, exact counts) and selected input manifest snapshots/provenance; `token_features.sqlite` (NPZ payload per sample, SHA256 checksum); `static_verification.json`; `training_checkpoint.json`, `best_weights.json`, `weights.json`, `training_history.json`; `frozen_weights.json`; `threshold.json`, `calibration_scores.parquet`; `heldout_adaptive_scores.parquet/csv`, `metrics.json`.

Token arrays are target-token aligned: input IDs, offsets, category IDs, raw z, sigmoid(z), original static weights, eligibility, attention/special masks, extraction-time static score and parser status. First/unpredicted, special and empty-offset targets have zero weight. Per-sample metadata carries exact IDs, labels, language, content hash and original role. No full vocabulary logits or model parameters are in the cache.

Scientific limits: reconstruction rather than recovered SERSEM author math; deterministic overlap attribution is ours; rare categories may remain near the regularized prior; fixed global weights need not improve AUC; 200 calibration examples give coarse low-FPR estimates; numerical fidelity, GPU fit and actual coverage must be validated on Kaggle. This notebook was developed with synthetic CPU tests; no target inference or model downloads were run locally.
